# 🚀 AllBGRemove - Ultra High Quality GPU Backend
This notebook runs your background removal API on a **FREE T4 GPU** from Google.

### 📋 Instructions (Beginner Friendly):
1. Go to **Runtime** > **Change runtime type** and ensure **T4 GPU** is selected.
2. Click the **Play button** (Run cell) on the block below.
3. Wait for the **Ngrok Public URL** to appear at the bottom.
4. Copy that URL and paste it in your frontend's `.env.local` as `NEXT_PUBLIC_API_URL`.

In [ ]:
# 1. Install Dependencies
!pip install rembg[gpu,onnxruntime] fastapi uvicorn python-multipart pyngrok pillow

# 2. Write the API Code (Premium 4K + Alpha Matting Edition)
with open('main.py', 'w') as f:
    f.write("""
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from rembg import remove, new_session
from PIL import Image
import io
import time
import os
import uuid
import asyncio
from fastapi.concurrency import run_in_threadpool

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

ai_session = None
tasks = {}
task_queue = asyncio.Queue()

def get_session():
    global ai_session
    if ai_session is None:
        print("Loading BiRefNet-General (Premium GPU Mode)...")
        ai_session = new_session("birefnet-general")
    return ai_session

@app.on_event("startup")
async def startup_event():
    asyncio.create_task(worker_loop())

async def worker_loop():
    while True:
        task = await task_queue.get()
        asyncio.create_task(process_task(task))

async def process_task(task):
    task_id, input_image, enhance, intensity = task
    tasks[task_id] = {"status": "processing"}
    try:
        # 4K Precision
        max_size = 4096 
        if input_image.size[0] > max_size or input_image.size[1] > max_size:
            input_image.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        
        start = time.time()
        def blocking_bg_removal():
            return remove(
                input_image, 
                session=get_session(),
                post_process_mask=True,
                alpha_matting=True,
                alpha_matting_foreground_threshold=240,
                alpha_matting_background_threshold=10,
                alpha_matting_erode_size=10
            ).convert("RGBA")
            
        output_image = await run_in_threadpool(blocking_bg_removal)
        
        os.makedirs("results", exist_ok=True)
        result_path = f"results/{task_id}.png"
        output_image.save(result_path, format='PNG')
        tasks[task_id]["status"] = "completed"
        tasks[task_id]["result_path"] = result_path
    except Exception as e:
        print(f"Error: {e}")
        tasks[task_id] = {"status": "failed", "error": str(e)}
    finally: task_queue.task_done()

@app.post("/api/remove-bg-async")
async def queue_remove_background(file: UploadFile = File(...), enhance: bool = False, intensity: float = 1.0):
    contents = await file.read()
    input_image = Image.open(io.BytesIO(contents)).convert("RGB")
    task_id = str(uuid.uuid4())
    await task_queue.put((task_id, input_image, enhance, intensity))
    return {"task_id": task_id}

@app.get("/api/status/{task_id}")
async def get_status(task_id: str):
    return tasks.get(task_id, {"status": "not_found"})

@app.get("/api/result/{task_id}")
async def get_result(task_id: str):
    path = f"results/{task_id}.png"
    if os.path.exists(path): return FileResponse(path, media_type="image/png")
    raise HTTPException(status_code=404)

""")

# 3. Expose with Ngrok
from pyngrok import ngrok
import uvicorn
import threading

# IMPORTANT: Insert your Ngrok Auth Token here if you have one
# ngrok.set_auth_token("YOUR_AUTH_TOKEN")

public_url = ngrok.connect(8000).public_url
print(f"\n🚀 CORE API IS READY!")
print(f"Copy this URL for your Frontend: {public_url}")
print(f"--------------------------------------------------")

uvicorn.run(app, host="0.0.0.0", port=8000)